###  Weather Data Aggregation (Daily → Monthly)

This notebook transforms daily weather data into monthly aggregates for each prefecture.

### Why?
Tourism demand is modeled at a monthly level. Daily fluctuations introduce noise, while monthly aggregation captures:
- Seasonal trends
- Climate patterns
- Demand-relevant variability

### Output:
data/weather_monthly/*.csv

In [ ]:
import pandas as pd 
import os 

In [ ]:
DAILY_DIR = "data/weather_daily_new_2026"
MONTHLY_DIR = "data/weather_monthly_new_1"

os.makedirs(MONTHLY_DIR, exist_ok=True)
MONTHLY_DIR


In [ ]:
def aggregate_daily_to_monthly(df_daily):
    # Ensure datetime
    df_daily["time"] = pd.to_datetime(df_daily["time"], dayfirst=True)

    # Extract month
    df_daily["month"] = df_daily["time"].dt.month

    # Aggregate features
    df_monthly = (
        df_daily
        .groupby(["prefecture", "year", "month"], as_index=False)
        .agg(
            temp_avg=("temperature_2m_mean", "mean"),
            temp_max=("temperature_2m_max", "max"),
            temp_min=("temperature_2m_min", "min"),
            precip_sum=("precipitation_sum", "sum"),
        )
    )

    # Round values for consistency
    numeric_cols = ["temp_avg", "temp_max", "temp_min", "precip_sum"]
    df_monthly[numeric_cols] = df_monthly[numeric_cols].round(2)

    return df_monthly

In [ ]:
for fname in os.listdir(DAILY_DIR):

    if not fname.endswith("_daily.csv"):
        continue

    prefecture = fname.replace("_daily.csv", "")

    input_path = os.path.join(DAILY_DIR, fname)
    output_path = os.path.join(MONTHLY_DIR, f"{prefecture}_monthly.csv")

    if os.path.exists(output_path):
        print(f"[SKIP] {prefecture}")
        continue

    df_daily = pd.read_csv(input_path)
    df_monthly = aggregate_daily_to_monthly(df_daily)

    df_monthly.to_csv(output_path, index=False)

    print(f"[SAVED] {prefecture}")

---
### Append 2026 Monthly Data to Existing Files

This section aggregates the 2026 daily weather data and **appends** the result
to the existing prefecture monthly CSVs in `data/weather_monthly_new_1/`.

- It will **skip** any prefecture whose file already contains year 2026 (safe to re-run).
- No existing data (2007–2025) is modified.

In [ ]:
# ─── APPEND 2026 DATA TO EXISTING MONTHLY FILES ───────────────────────────
# Reads from : data/weather_daily_new_2026  (raw daily, Jan–Jun 2026)
# Appends to : data/weather_monthly_new_1   (existing 2007–2025 monthly)
# Safe to re-run: skips prefecture if 2026 rows already exist in the file.
# ──────────────────────────────────────────────────────────────────────────

DAILY_2026_DIR  = "data/weather_daily_new_2026"
MONTHLY_OUT_DIR = "data/weather_monthly_new_1"

for fname in sorted(os.listdir(DAILY_2026_DIR)):

    if not fname.endswith("_daily.csv"):
        continue

    prefecture = fname.replace("_daily.csv", "")
    monthly_path = os.path.join(MONTHLY_OUT_DIR, f"{prefecture}_monthly.csv")

    # ── Safety check: target monthly file must already exist ──────────────
    if not os.path.exists(monthly_path):
        print(f"[MISSING] {prefecture} — monthly file not found, skipping.")
        continue

    # ── Load existing monthly data ────────────────────────────────────────
    df_existing = pd.read_csv(monthly_path)

    # ── Duplicate guard: skip if 2026 rows already present ────────────────
    if 2026 in df_existing["year"].values:
        print(f"[SKIP] {prefecture} — 2026 data already exists.")
        continue

    # ── Load & aggregate 2026 daily data ──────────────────────────────────
    daily_path = os.path.join(DAILY_2026_DIR, fname)
    df_daily_2026 = pd.read_csv(daily_path)

    # Drop any fully empty rows (trailing blank lines in source CSV)
    df_daily_2026.dropna(how="all", inplace=True)

    df_monthly_2026 = aggregate_daily_to_monthly(df_daily_2026)

    # ── Append 2026 rows to existing data ─────────────────────────────────
    df_combined = pd.concat([df_existing, df_monthly_2026], ignore_index=True)

    # ── Sort by year then month for clean ordering ────────────────────────
    df_combined.sort_values(["year", "month"], inplace=True)
    df_combined.reset_index(drop=True, inplace=True)

    # ── Save back to the same file ────────────────────────────────────────
    df_combined.to_csv(monthly_path, index=False)

    print(f"[APPENDED] {prefecture} — {len(df_monthly_2026)} rows of 2026 data added.")

print("\nDone! All 2026 monthly data appended.")
